# AgriMind — ML pipeline on Google Colab

This notebook runs the **same training pipeline** as your local backend (`python -m ml_models.pipeline.run_all`):
- Downloads real APY-style crop data (or uses your CSV)
- Trains yield regressors + crop classifier
- Saves artifacts under `backend/ml_models/artifacts/`

**Note:** The full Flask + React app is not run here; only the **data + ML pipeline**. For the API, use your machine or deploy separately.

**GPU:** Training uses CPU-friendly sklearn/XGBoost; GPU is optional and not required.

---
### If `list(Path('/content').iterdir())` shows only `.config`, `data`, `sample_data`
That means **your project code is not on this runtime yet**. You must run **one** of:
- **Section 2** (git clone), or
- **Section 2b** (download GitHub as ZIP — no git), or
- **Section 2c** (upload ZIP from your laptop).

Then run **section 3** again.

## 1) Install dependencies (minimal for the pipeline)
Full `requirements.txt` includes Flask, LangChain, etc. For Colab training we only need these.

In [ ]:
%%capture
!pip install -q numpy pandas scikit-learn xgboost joblib shap matplotlib seaborn

## 2) Get your project code

**Option A — GitHub (replace with your repo URL):**

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/MLproject.git"  # <-- MUST be your real GitHub URL
BRANCH = "main"

%cd /content
!rm -rf MLproject
!git clone -b {BRANCH} --depth 1 {REPO_URL} MLproject

# Git names the folder "MLproject" only if the URL ends with MLproject.git
# If clone used another repo name, you will see that folder under /content instead.
print("\n=== Folders in /content ===")
for p in sorted(Path("/content").iterdir()):
    print(" ", p.name, "(dir)" if p.is_dir() else "")

candidates = [
    Path("/content/MLproject/backend"),
    Path("/content/MLproject"),  # rare: backend files at repo root
]
found = None
for c in candidates:
    if (c / "ml_models").is_dir() and (c / "app.py").is_file():
        found = c
        break
if found:
    print("\nOK: backend found at", found)
else:
    print("\n!!! Clone may have failed OR repo layout differs. See section 3 to auto-search.")
    print("Check: is REPO_URL correct? Is the repo public? Any error above from git clone?")

**Option B — Upload a ZIP of the project** (if the repo is private):
1. Zip your `MLproject` folder locally.
2. Run the cell below, choose the zip, then adjust `ZIP_NAME` if needed.

In [ ]:
# See section 2c below for upload.

### 2b) Download project as ZIP from GitHub (no `git` — **public repo only**)

Set `GITHUB_USER` and `REPO_NAME` to match your repository. GitHub names the folder **`<repo>-main`** using your repo name (e.g. `MLproject-main` or `ml-project-main`). Section 3 searches common variants automatically.

In [ ]:
from pathlib import Path
import urllib.request
import zipfile

GITHUB_USER = "YOUR_USERNAME"  # <-- e.g. pranjali-yeotikar
REPO_NAME = "MLproject"          # <-- your repo name
BRANCH = "main"                 # or master

root = Path("/content")
zip_path = root / "repo_source.zip"
url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}/archive/refs/heads/{BRANCH}.zip"

print("Downloading:", url)
urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(root)
print("Done. Folders in /content:")
for p in sorted(root.iterdir()):
    if p.is_dir() and not p.name.startswith("."):
        print(" ", p.name)

### 2c) Upload ZIP from your computer (private repo or no GitHub)

On your Mac: zip the **`MLproject`** folder (the one that contains the `backend` folder). Run the cell and select that zip.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()  # pick MLproject.zip
root = Path("/content")
for name in uploaded:
    zip_path = root / name
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(root)
    print("Extracted:", name)
print("Folders:", [p.name for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")])

## 3) Point Python at the `backend` folder (fixes `No module named 'ml_models'`)

Colab often resets the working directory to `/content`. The package `ml_models` lives **inside** `MLproject/backend`, so we **must** add that folder to `sys.path` and `chdir` there before importing.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"


def find_backend() -> Path:
    """Directory that contains both `app.py` and `ml_models/` (Flask backend root)."""
    flat = [
        Path("/content/MLproject/backend"),
        Path("/content/MLproject-main/backend"),   # GitHub ZIP (repo name MLproject)
        Path("/content/ml-project-main/backend"),  # GitHub ZIP (repo name ml-project)
        Path("/content/MLproject-master/backend"),
        Path("/content/MLproject"),
        Path("/content/MLproject-main"),
        Path("/content/ml-project-main"),
        Path("/content/backend"),
        Path.cwd(),
    ]
    for c in flat:
        try:
            if c.is_dir() and (c / "ml_models").is_dir() and (c / "app.py").is_file():
                return c.resolve()
        except OSError:
            pass
    # Any repo under /content: .../<repo>/backend OR .../<repo> is the backend
    root = Path("/content")
    if root.is_dir():
        for child in sorted(root.iterdir()):
            if not child.is_dir() or child.name.startswith("."):
                continue
            for sub in (child / "backend", child):
                if (
                    sub.is_dir()
                    and (sub / "ml_models").is_dir()
                    and (sub / "app.py").is_file()
                ):
                    return sub.resolve()
        # Deep search (nested zips, odd layouts)
        for app in root.rglob("app.py"):
            d = app.parent
            if (d / "ml_models").is_dir():
                return d.resolve()
    raise FileNotFoundError(
        "No folder with app.py + ml_models/ under /content.\n"
        "Fix: (1) Run git clone with a valid REPO_URL, or (2) upload ZIP and unzip under /content,\n"
        "or (3) set BACKEND_MANUAL below to your backend path."
    )


# If auto-detect fails, set this to a string path you see in Colab's file browser, e.g.:
# BACKEND_MANUAL = "/content/MyFolderName/backend"
BACKEND_MANUAL = None


if BACKEND_MANUAL:
    BACKEND = Path(BACKEND_MANUAL).resolve()
    if not ((BACKEND / "ml_models").is_dir() and (BACKEND / "app.py").is_file()):
        raise FileNotFoundError(BACKEND)
else:
    BACKEND = find_backend()
os.chdir(BACKEND)
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
print("BACKEND =", BACKEND)
print("sys.path[0] =", sys.path[0])

## 4) Download crop production CSV (if missing)
Same source as `backend/scripts/download_real_crop_data.py`. Uses a subsample for faster Colab runs; set `MAX_ROWS = 0` in the URL path logic below for full file (~345k rows, slower).

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
csv_path = DATA_DIR / "crop_production_india.csv"

URL = "https://raw.githubusercontent.com/vish-manit/Crop-Production-Statistics-in-India/master/APY.csv"
MAX_ROWS = 50000  # increase or set None for full dataset

if not csv_path.is_file():
    print("Downloading APY data...")
    df = pd.read_csv(URL, low_memory=False)
    if MAX_ROWS and len(df) > MAX_ROWS:
        df = df.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)
    df.to_csv(csv_path, index=False)
    print("Saved", len(df), "rows ->", csv_path)
else:
    print("Using existing", csv_path)

## 5) Optional — climate normals file
If your repo already contains `data/india_state_climate_normals.csv`, skip this. Otherwise clone should have included it.

In [ ]:
from pathlib import Path
p = Path("data/india_state_climate_normals.csv")
print("Climate file exists:", p.is_file(), p)

## 6) Run the full training pipeline
This is equivalent to:
```bash
cd backend
export AGRIMIND_MAX_TRAIN_ROWS=50000   # optional cap
python -m ml_models.pipeline.run_all
```

In [ ]:
import os
import sys
from pathlib import Path

def _discover_backend():
    for c in [
        Path("/content/MLproject/backend"),
        Path("/content/MLproject-main/backend"),
        Path("/content/ml-project-main/backend"),
        Path("/content/MLproject-master/backend"),
        Path("/content/MLproject"),
        Path("/content/MLproject-main"),
        Path("/content/ml-project-main"),
        Path("/content/backend"),
        Path.cwd(),
    ]:
        try:
            if c.is_dir() and (c / "ml_models").is_dir() and (c / "app.py").is_file():
                return c.resolve()
        except OSError:
            pass
    root = Path("/content")
    if root.is_dir():
        for child in sorted(root.iterdir()):
            if not child.is_dir() or child.name.startswith("."):
                continue
            for sub in (child / "backend", child):
                if (
                    sub.is_dir()
                    and (sub / "ml_models").is_dir()
                    and (sub / "app.py").is_file()
                ):
                    return sub.resolve()
        for app in root.rglob("app.py"):
            d = app.parent
            if (d / "ml_models").is_dir():
                return d.resolve()
    return None

b = _discover_backend()
if b is None:
    raise FileNotFoundError(
        "Backend not found. Run git clone (section 2) or upload ZIP. "
        \"List /content: \" + str(list(Path('/content').iterdir()))\n
    )
os.chdir(b)
if str(b) not in sys.path:
    sys.path.insert(0, str(b))
print("Using backend:", b)


os.environ["AGRIMIND_MAX_TRAIN_ROWS"] = "50000"
os.environ.pop("AGRIMIND_USE_SYNTHETIC_DATA", None)

from ml_models.pipeline.run_all import main
main()

## 7) Show saved artifacts

In [ ]:
from pathlib import Path
art = Path("ml_models/artifacts")
for f in sorted(art.rglob("*")):
    if f.is_file():
        print(f.stat().st_size // 1024, "KB", f.relative_to(art))

## 8) Quick inference demo (single prediction)

In [ ]:
import joblib
import pandas as pd

pipe = joblib.load("ml_models/artifacts/yield_xgb.joblib")

def rainfall_cat(r):
    r = float(r)
    if r < 500: return "low"
    if r <= 1000: return "medium"
    return "high"

row = pd.DataFrame([{
    "state": "maharashtra",
    "crop": "rice",
    "season": "kharif",
    "rainfall": 1100.0,
    "temperature": 27.0,
    "area": 2.0,
    "rainfall_category": rainfall_cat(1100.0),
}])
y_hat = float(pipe.predict(row)[0])
print("Predicted yield (t/ha):", round(y_hat, 3))

## 9) Download artifacts to your computer
Useful if you want to copy `yield_xgb.joblib` back into your local `backend/ml_models/artifacts/`.

In [ ]:
from google.colab import files
# Example: download yield bundle only
# files.download('ml_models/artifacts/yield_xgb.joblib')

---
### Optional: Prophet (forecast artifact)
Step 6 already tries Prophet at the end; if it fails on Colab, you can install:
`!pip install -q prophet` and re-run, or ignore — the yield pipeline does not depend on it.

### Running the Flask API on Colab
Possible with `pyngrok`, but not recommended for production. For coursework demos only:
`!pip install pyngrok flask flask-cors` then run `app.py` and expose port 5000 via ngrok token.